# Feature Engineering Notebook

This notebook loads OULAD-style CSV tables, cleans and merges data, and engineers features for both supervised and unsupervised learning tasks.

**Outputs:**
- `./data/features_supervised.parquet` - Features with labels (final_result, at_risk)
- `./data/features_unsupervised.parquet` - Features without labels
- `./reports/feature_dictionary.csv` - Data dictionary


## 0. Setup


In [18]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Create directories if they don't exist
os.makedirs('./data', exist_ok=True)
os.makedirs('./reports', exist_ok=True)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("Setup complete.")


Setup complete.


## 1. Data Loading


In [19]:
# Load all CSV files
print("Loading data files...")

student_info = pd.read_csv('./data/studentInfo.csv')
print(f"studentInfo: {student_info.shape}")

assessments = pd.read_csv('./data/assessments.csv')
print(f"assessments: {assessments.shape}")

student_assessment = pd.read_csv('./data/studentAssessment.csv')
print(f"studentAssessment: {student_assessment.shape}")

# Load VLE data (may be large)
print("Loading VLE data (this may take a moment)...")
student_vle = pd.read_csv('./data/studentVle.csv')
print(f"studentVle: {student_vle.shape}")

vle = pd.read_csv('./data/vle.csv')
print(f"vle: {vle.shape}")

print("\nData loading complete.")


Loading data files...
studentInfo: (32593, 12)
assessments: (206, 6)
studentAssessment: (173912, 5)
Loading VLE data (this may take a moment)...
studentVle: (10655280, 6)
vle: (6364, 6)

Data loading complete.


## 2. Data Understanding (Quality Checks)


In [20]:
# Check studentInfo structure
print("=== studentInfo ===")
print(f"Shape: {student_info.shape}")
print(f"\nColumns: {list(student_info.columns)}")
print(f"\nUnique students: {student_info['id_student'].nunique()}")
print(f"Unique (student, module, presentation): {student_info.groupby(['id_student', 'code_module', 'code_presentation']).size().shape[0]}")
print(f"\nMissing values:\n{student_info.isnull().sum()}")
print(f"\nfinal_result values:\n{student_info['final_result'].value_counts()}")


=== studentInfo ===
Shape: (32593, 12)

Columns: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result']

Unique students: 28785
Unique (student, module, presentation): 32593

Missing values:
code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

final_result values:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64


In [21]:
# Check assessments structure
print("=== assessments ===")
print(f"Shape: {assessments.shape}")
print(f"\nColumns: {list(assessments.columns)}")
print(f"\nMissing values:\n{assessments.isnull().sum()}")
print(f"\nAssessment types:\n{assessments['assessment_type'].value_counts()}")


=== assessments ===
Shape: (206, 6)

Columns: ['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']

Missing values:
code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

Assessment types:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64


In [22]:
# Check studentAssessment structure
print("=== studentAssessment ===")
print(f"Shape: {student_assessment.shape}")
print(f"\nColumns: {list(student_assessment.columns)}")
print(f"\nMissing values:\n{student_assessment.isnull().sum()}")
print(f"\nSample of score distribution:\n{student_assessment['score'].describe()}")


=== studentAssessment ===
Shape: (173912, 5)

Columns: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']

Missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

Sample of score distribution:
count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64


In [23]:
# Check VLE structure
print("=== vle ===")
print(f"Shape: {vle.shape}")
print(f"\nColumns: {list(vle.columns)}")
print(f"\nActivity types:\n{vle['activity_type'].value_counts()}")

print("\n=== studentVle ===")
print(f"Shape: {student_vle.shape}")
print(f"\nColumns: {list(student_vle.columns)}")
print(f"\nMissing values:\n{student_vle.isnull().sum()}")
print(f"\nSample of sum_click distribution:\n{student_vle['sum_click'].describe()}")


=== vle ===
Shape: (6364, 6)

Columns: ['id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']

Activity types:
activity_type
resource          2660
subpage           1055
oucontent          996
url                886
forumng            194
quiz               127
page               102
oucollaborate       82
questionnaire       61
ouwiki              49
dataplus            28
externalquiz        26
homepage            22
ouelluminate        21
glossary            21
dualpane            20
repeatactivity       5
htmlactivity         4
sharedsubpage        3
folder               2
Name: count, dtype: int64

=== studentVle ===
Shape: (10655280, 6)

Columns: ['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']

Missing values:
code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

Sample of sum_click distribution:
count    1.065528e

## 3. Cleaning (Types + Missing Values)


In [ ]:
# Clean studentInfo
print("Cleaning studentInfo...")
student_info_clean = student_info.copy()

# Impute missing imd_band values with mode per region
imd_mode_by_region = (
    student_info_clean
    .dropna(subset=["imd_band"])
    .groupby("region")["imd_band"]
    .agg(lambda x: x.mode().iloc[0])
)

student_info_clean["imd_band"] = student_info_clean["imd_band"].fillna(
    student_info_clean["region"].map(imd_mode_by_region)
)

# Map gender to binary values
gender_mapping = {
    "M": 0,
    "F": 1
}

student_info_clean["gender_bin"] = student_info_clean["gender"].map(gender_mapping)

# Map highest_education to ordinal values
education_mapping = {
    "No Formal quals": 0,
    "Lower Than A Level": 1,
    "A Level or Equivalent": 2,
    "HE Qualification": 3,
    "Post Graduate Qualification": 4
}

student_info_clean["highest_education_ord"] = student_info_clean["highest_education"].map(education_mapping)

# Map imd_band to ordinal values
imd_mapping = {
    "0-10%": 0,
    "10-20%": 1,
    "20-30%": 2,
    "30-40%": 3,
    "40-50%": 4,
    "50-60%": 5,
    "60-70%": 6,
    "70-80%": 7,
    "80-90%": 8,
    "90-100%": 9
}

student_info_clean["imd_band_ord"] = student_info_clean["imd_band"].map(imd_mapping)

# Map age_band to ordinal values
age_mapping = {
    "0-35": 0,
    "35-55": 1,
    "55<=": 2
}

student_info_clean["Age_band_ord"] = student_info_clean["age_band"].map(age_mapping)

# Map disability to binary values
disability_mapping = {
    "N": 0,
    "Y": 1
}

student_info_clean["disability_bin"] = student_info_clean["disability"].map(disability_mapping)

# Convert date columns if they exist (none in studentInfo)
# Ensure key columns are correct types
student_info_clean['id_student'] = student_info_clean['id_student'].astype(str)
student_info_clean['code_module'] = student_info_clean['code_module'].astype(str)
student_info_clean['code_presentation'] = student_info_clean['code_presentation'].astype(str)

# Convert numeric columns
numeric_cols = ['num_of_prev_attempts', 'studied_credits']
for col in numeric_cols:
    if col in student_info_clean.columns:
        student_info_clean[col] = pd.to_numeric(student_info_clean[col], errors='coerce')

print(f"studentInfo cleaned: {student_info_clean.shape}")


Cleaning studentInfo...
studentInfo cleaned: (32593, 12)


In [25]:
# Clean assessments
print("Cleaning assessments...")
assessments_clean = assessments.copy()

# Convert key columns
assessments_clean['code_module'] = assessments_clean['code_module'].astype(str)
assessments_clean['code_presentation'] = assessments_clean['code_presentation'].astype(str)
assessments_clean['id_assessment'] = assessments_clean['id_assessment'].astype(str)

# Convert date and weight to numeric
assessments_clean['date'] = pd.to_numeric(assessments_clean['date'], errors='coerce')
assessments_clean['weight'] = pd.to_numeric(assessments_clean['weight'], errors='coerce')

print(f"assessments cleaned: {assessments_clean.shape}")


Cleaning assessments...
assessments cleaned: (206, 6)


In [26]:
# Clean studentAssessment
print("Cleaning studentAssessment...")
student_assessment_clean = student_assessment.copy()

# Convert key columns
student_assessment_clean['id_assessment'] = student_assessment_clean['id_assessment'].astype(str)
student_assessment_clean['id_student'] = student_assessment_clean['id_student'].astype(str)

# Convert date and score to numeric
student_assessment_clean['date_submitted'] = pd.to_numeric(student_assessment_clean['date_submitted'], errors='coerce')
student_assessment_clean['score'] = pd.to_numeric(student_assessment_clean['score'], errors='coerce')
student_assessment_clean['is_banked'] = pd.to_numeric(student_assessment_clean['is_banked'], errors='coerce')

print(f"studentAssessment cleaned: {student_assessment_clean.shape}")


Cleaning studentAssessment...
studentAssessment cleaned: (173912, 5)


In [27]:
# Clean vle
print("Cleaning vle...")
vle_clean = vle.copy()

# Convert key columns
vle_clean['id_site'] = vle_clean['id_site'].astype(str)
vle_clean['code_module'] = vle_clean['code_module'].astype(str)
vle_clean['code_presentation'] = vle_clean['code_presentation'].astype(str)

# Convert week columns to numeric
vle_clean['week_from'] = pd.to_numeric(vle_clean['week_from'], errors='coerce')
vle_clean['week_to'] = pd.to_numeric(vle_clean['week_to'], errors='coerce')

print(f"vle cleaned: {vle_clean.shape}")


Cleaning vle...
vle cleaned: (6364, 6)


In [28]:
# Clean studentVle
print("Cleaning studentVle...")
student_vle_clean = student_vle.copy()

# Convert key columns
student_vle_clean['id_student'] = student_vle_clean['id_student'].astype(str)
student_vle_clean['id_site'] = student_vle_clean['id_site'].astype(str)
student_vle_clean['code_module'] = student_vle_clean['code_module'].astype(str)
student_vle_clean['code_presentation'] = student_vle_clean['code_presentation'].astype(str)

# Convert date and sum_click to appropriate types
if 'date' in student_vle_clean.columns:
    student_vle_clean['date'] = pd.to_numeric(student_vle_clean['date'], errors='coerce')
student_vle_clean['sum_click'] = pd.to_numeric(student_vle_clean['sum_click'], errors='coerce')

print(f"studentVle cleaned: {student_vle_clean.shape}")
print("Cleaning complete.")


Cleaning studentVle...
studentVle cleaned: (10655280, 6)
Cleaning complete.


## 4. Join & Master Table Creation


In [29]:
# Start with studentInfo as base table (one row per student-module-presentation)
master = student_info_clean.copy()

print(f"Master table initial shape: {master.shape}")
print(f"Master table key columns: {['id_student', 'code_module', 'code_presentation']}")

# Verify no duplicates on key
duplicates = master.duplicated(subset=['id_student', 'code_module', 'code_presentation']).sum()
print(f"Duplicate keys: {duplicates}")
assert duplicates == 0, "Master table has duplicate keys!"


Master table initial shape: (32593, 12)
Master table key columns: ['id_student', 'code_module', 'code_presentation']
Duplicate keys: 0


## 5. Feature Engineering (Aggregation)


In [30]:
# Merge assessments with studentAssessment to get per-student assessment data
print("Merging assessment data...")

# Join assessments and studentAssessment on id_assessment
assessment_merged = student_assessment_clean.merge(
    assessments_clean,
    on='id_assessment',
    how='left',
    suffixes=('', '_assess')
)

print(f"Assessment merged shape: {assessment_merged.shape}")

# Filter to only records matching master table keys
assessment_merged = assessment_merged.merge(
    master[['id_student', 'code_module', 'code_presentation']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='inner'
)

print(f"Assessment merged (filtered to master keys): {assessment_merged.shape}")


Merging assessment data...
Assessment merged shape: (173912, 10)
Assessment merged (filtered to master keys): (173912, 10)


In [31]:
# Create assessment features per student-module-presentation
print("Creating assessment features...")

assessment_features = assessment_merged.groupby(['id_student', 'code_module', 'code_presentation']).agg({
    'score': ['mean', 'max', 'count'],
    'date_submitted': 'count',
    'id_assessment': 'count'
}).reset_index()

# Flatten column names
assessment_features.columns = ['id_student', 'code_module', 'code_presentation', 
                                'mean_score', 'max_score', 'score_count', 
                                'submissions_count', 'assessments_count']

# Calculate missed submissions
total_assessments = assessment_merged.groupby(['id_student', 'code_module', 'code_presentation'])['id_assessment'].nunique().reset_index()
total_assessments.columns = ['id_student', 'code_module', 'code_presentation', 'total_assessments_available']

assessment_features = assessment_features.merge(total_assessments, on=['id_student', 'code_module', 'code_presentation'], how='left')
assessment_features['missed_submissions_count'] = assessment_features['total_assessments_available'] - assessment_features['submissions_count']

# Calculate submission delay (date_submitted - date)
if 'date' in assessment_merged.columns and 'date_submitted' in assessment_merged.columns:
    assessment_merged['submission_delay'] = assessment_merged['date_submitted'] - assessment_merged['date']
    avg_delay = assessment_merged.groupby(['id_student', 'code_module', 'code_presentation'])['submission_delay'].mean().reset_index()
    avg_delay.columns = ['id_student', 'code_module', 'code_presentation', 'avg_submission_delay']
    assessment_features = assessment_features.merge(avg_delay, on=['id_student', 'code_module', 'code_presentation'], how='left')
else:
    assessment_features['avg_submission_delay'] = np.nan

print(f"Assessment features shape: {assessment_features.shape}")
print(f"Assessment features columns: {list(assessment_features.columns)}")


Creating assessment features...
Assessment features shape: (25843, 11)
Assessment features columns: ['id_student', 'code_module', 'code_presentation', 'mean_score', 'max_score', 'score_count', 'submissions_count', 'assessments_count', 'total_assessments_available', 'missed_submissions_count', 'avg_submission_delay']


In [32]:
# Merge studentVle with vle to get activity type information
print("Merging VLE data...")

vle_merged = student_vle_clean.merge(
    vle_clean,
    on=['id_site', 'code_module', 'code_presentation'],
    how='left',
    suffixes=('', '_vle')
)

print(f"VLE merged shape: {vle_merged.shape}")

# Filter to only records matching master table keys
vle_merged = vle_merged.merge(
    master[['id_student', 'code_module', 'code_presentation']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='inner'
)

print(f"VLE merged (filtered to master keys): {vle_merged.shape}")


Merging VLE data...
VLE merged shape: (10655280, 9)
VLE merged (filtered to master keys): (10655280, 9)


In [33]:
# Create VLE engagement features per student-module-presentation
print("Creating VLE engagement features...")

# Total clicks
vle_total_clicks = vle_merged.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
vle_total_clicks.columns = ['id_student', 'code_module', 'code_presentation', 'total_clicks']

# Active days (count distinct dates)
if 'date' in vle_merged.columns:
    vle_active_days = vle_merged.groupby(['id_student', 'code_module', 'code_presentation'])['date'].nunique().reset_index()
    vle_active_days.columns = ['id_student', 'code_module', 'code_presentation', 'active_days']
else:
    # If no date column, create a placeholder
    vle_active_days = master[['id_student', 'code_module', 'code_presentation']].copy()
    vle_active_days['active_days'] = np.nan

# Clicks by activity type (pivot)
if 'activity_type' in vle_merged.columns:
    vle_by_type = vle_merged.groupby(['id_student', 'code_module', 'code_presentation', 'activity_type'])['sum_click'].sum().reset_index()
    vle_pivot = vle_by_type.pivot_table(
        index=['id_student', 'code_module', 'code_presentation'],
        columns='activity_type',
        values='sum_click',
        fill_value=0
    ).reset_index()
    vle_pivot.columns.name = None
    # Rename columns to include prefix
    vle_pivot.columns = ['id_student', 'code_module', 'code_presentation'] + [f'clicks_{col}' for col in vle_pivot.columns[3:]]
else:
    vle_pivot = master[['id_student', 'code_module', 'code_presentation']].copy()

# Merge all VLE features
vle_features = vle_total_clicks.merge(vle_active_days, on=['id_student', 'code_module', 'code_presentation'], how='outer')
vle_features = vle_features.merge(vle_pivot, on=['id_student', 'code_module', 'code_presentation'], how='outer')

print(f"VLE features shape: {vle_features.shape}")
print(f"VLE features columns: {list(vle_features.columns)}")


Creating VLE engagement features...
VLE features shape: (29228, 25)
VLE features columns: ['id_student', 'code_module', 'code_presentation', 'total_clicks', 'active_days', 'clicks_dataplus', 'clicks_dualpane', 'clicks_externalquiz', 'clicks_folder', 'clicks_forumng', 'clicks_glossary', 'clicks_homepage', 'clicks_htmlactivity', 'clicks_oucollaborate', 'clicks_oucontent', 'clicks_ouelluminate', 'clicks_ouwiki', 'clicks_page', 'clicks_questionnaire', 'clicks_quiz', 'clicks_repeatactivity', 'clicks_resource', 'clicks_sharedsubpage', 'clicks_subpage', 'clicks_url']


In [ ]:
# Create time bucket features if date exists
if 'date' in vle_merged.columns:
    print("Creating time bucket features...")
    
    # Define time buckets based on date distribution
    # We'll use percentiles to define early, mid, late periods
    date_stats = vle_merged.groupby(['code_module', 'code_presentation'])['date'].agg(['min', 'max']).reset_index()
    date_stats['date_range'] = date_stats['max'] - date_stats['min']
    date_stats['early_threshold'] = date_stats['min'] + date_stats['date_range'] * 0.33
    date_stats['mid_threshold'] = date_stats['min'] + date_stats['date_range'] * 0.67
    
    # Merge thresholds back 
    vle_with_buckets = vle_merged.merge(date_stats[['code_module', 'code_presentation', 'early_threshold', 'mid_threshold']],
                                       on=['code_module', 'code_presentation'], how='left')
    
    # Assign buckets
    vle_with_buckets['time_bucket'] = 'mid'
    vle_with_buckets.loc[vle_with_buckets['date'] <= vle_with_buckets['early_threshold'], 'time_bucket'] = 'early'
    vle_with_buckets.loc[vle_with_buckets['date'] > vle_with_buckets['mid_threshold'], 'time_bucket'] = 'late'
    
    # Aggregate clicks by time bucket
    time_bucket_features = vle_with_buckets.groupby(['id_student', 'code_module', 'code_presentation', 'time_bucket'])['sum_click'].sum().reset_index()
    time_bucket_pivot = time_bucket_features.pivot_table(
        index=['id_student', 'code_module', 'code_presentation'],
        columns='time_bucket',
        values='sum_click',
        fill_value=0
    ).reset_index()
    time_bucket_pivot.columns.name = None
    time_bucket_pivot.columns = ['id_student', 'code_module', 'code_presentation', 'clicks_early', 'clicks_mid', 'clicks_late']
    
    # Merge with VLE features
    vle_features = vle_features.merge(time_bucket_pivot, on=['id_student', 'code_module', 'code_presentation'], how='left')
    
    print(f"Time bucket features added. VLE features shape: {vle_features.shape}")
else:
    print("No date column in VLE data, skipping time bucket features.")
    vle_features['clicks_early'] = np.nan
    vle_features['clicks_mid'] = np.nan
    vle_features['clicks_late'] = np.nan


Creating time bucket features...
Time bucket features added. VLE features shape: (29228, 28)


In [35]:
# Merge all features with master table
print("Merging all features...")

features = master.merge(
    assessment_features,
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
).merge(
    vle_features,
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

print(f"Final features shape: {features.shape}")
print(f"Final features columns: {list(features.columns)}")


Merging all features...
Final features shape: (32593, 45)
Final features columns: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'mean_score', 'max_score', 'score_count', 'submissions_count', 'assessments_count', 'total_assessments_available', 'missed_submissions_count', 'avg_submission_delay', 'total_clicks', 'active_days', 'clicks_dataplus', 'clicks_dualpane', 'clicks_externalquiz', 'clicks_folder', 'clicks_forumng', 'clicks_glossary', 'clicks_homepage', 'clicks_htmlactivity', 'clicks_oucollaborate', 'clicks_oucontent', 'clicks_ouelluminate', 'clicks_ouwiki', 'clicks_page', 'clicks_questionnaire', 'clicks_quiz', 'clicks_repeatactivity', 'clicks_resource', 'clicks_sharedsubpage', 'clicks_subpage', 'clicks_url', 'clicks_early', 'clicks_mid', 'clicks_late']


In [36]:
# Create at_risk label from final_result
print("Creating at_risk label...")

# Check unique final_result values
unique_results = features['final_result'].unique()
print(f"Unique final_result values: {unique_results}")

# Map to binary at_risk label
# at_risk = 1 if final_result in ["Fail","Withdrawn"]
# at_risk = 0 if final_result in ["Pass","Distinction"]
features['at_risk'] = 0
features.loc[features['final_result'].isin(['Fail', 'Withdrawn']), 'at_risk'] = 1

# Handle any other values (if they exist)
other_values = [v for v in unique_results if v not in ['Fail', 'Withdrawn', 'Pass', 'Distinction']]
if len(other_values) > 0:
    print(f"Warning: Found additional final_result values: {other_values}")
    # Map based on typical interpretation: anything not Pass/Distinction is at risk
    for val in other_values:
        if val not in ['Pass', 'Distinction']:
            features.loc[features['final_result'] == val, 'at_risk'] = 1

print(f"\nat_risk distribution:\n{features['at_risk'].value_counts()}")
print(f"\nfinal_result vs at_risk cross-tab:\n{pd.crosstab(features['final_result'], features['at_risk'])}")


Creating at_risk label...
Unique final_result values: ['Pass' 'Withdrawn' 'Fail' 'Distinction']

at_risk distribution:
at_risk
1    17208
0    15385
Name: count, dtype: int64

final_result vs at_risk cross-tab:
at_risk           0      1
final_result              
Distinction    3024      0
Fail              0   7052
Pass          12361      0
Withdrawn         0  10156


## 6. Feature Validation (Sanity Checks)


In [37]:
# Create supervised features (with labels)
features_supervised = features.copy()

print("=== SUPERVISED FEATURES ===")
print(f"Shape: {features_supervised.shape}")
print(f"Key columns: {['id_student', 'code_module', 'code_presentation']}")

# Check for duplicates
duplicates_sup = features_supervised.duplicated(subset=['id_student', 'code_module', 'code_presentation']).sum()
print(f"Duplicate keys: {duplicates_sup}")
assert duplicates_sup == 0, "Supervised features have duplicate keys!"

# Check for labels
print(f"\nHas final_result: {'final_result' in features_supervised.columns}")
print(f"Has at_risk: {'at_risk' in features_supervised.columns}")

# Class balance
if 'at_risk' in features_supervised.columns:
    print(f"\nClass balance (at_risk):")
    print(features_supervised['at_risk'].value_counts(normalize=True))


=== SUPERVISED FEATURES ===
Shape: (32593, 46)
Key columns: ['id_student', 'code_module', 'code_presentation']
Duplicate keys: 0

Has final_result: True
Has at_risk: True

Class balance (at_risk):
at_risk
1    0.527966
0    0.472034
Name: proportion, dtype: float64


In [38]:
# Create unsupervised features (without labels)
features_unsupervised = features.drop(columns=['final_result', 'at_risk'], errors='ignore').copy()

print("\n=== UNSUPERVISED FEATURES ===")
print(f"Shape: {features_unsupervised.shape}")

# Check for duplicates
duplicates_unsup = features_unsupervised.duplicated(subset=['id_student', 'code_module', 'code_presentation']).sum()
print(f"Duplicate keys: {duplicates_unsup}")
assert duplicates_unsup == 0, "Unsupervised features have duplicate keys!"

# Verify labels are removed
print(f"\nHas final_result: {'final_result' in features_unsupervised.columns}")
print(f"Has at_risk: {'at_risk' in features_unsupervised.columns}")
assert 'final_result' not in features_unsupervised.columns, "final_result should be removed!"
assert 'at_risk' not in features_unsupervised.columns, "at_risk should be removed!"



=== UNSUPERVISED FEATURES ===
Shape: (32593, 44)
Duplicate keys: 0

Has final_result: False
Has at_risk: False


In [39]:
# Missingness summary
print("\n=== MISSINGNESS SUMMARY ===")
print("\nSupervised features missingness:")
missing_sup = features_supervised.isnull().sum()
missing_sup_pct = (missing_sup / len(features_supervised) * 100).round(2)
missing_df_sup = pd.DataFrame({
    'missing_count': missing_sup,
    'missing_pct': missing_sup_pct
})
missing_df_sup = missing_df_sup[missing_df_sup['missing_count'] > 0].sort_values('missing_count', ascending=False)
print(missing_df_sup)

print("\nUnsupervised features missingness:")
missing_unsup = features_unsupervised.isnull().sum()
missing_unsup_pct = (missing_unsup / len(features_unsupervised) * 100).round(2)
missing_df_unsup = pd.DataFrame({
    'missing_count': missing_unsup,
    'missing_pct': missing_unsup_pct
})
missing_df_unsup = missing_df_unsup[missing_df_unsup['missing_count'] > 0].sort_values('missing_count', ascending=False)
print(missing_df_unsup)



=== MISSINGNESS SUMMARY ===

Supervised features missingness:
                             missing_count  missing_pct
max_score                             6773        20.78
mean_score                            6773        20.78
avg_submission_delay                  6751        20.71
score_count                           6750        20.71
submissions_count                     6750        20.71
assessments_count                     6750        20.71
total_assessments_available           6750        20.71
missed_submissions_count              6750        20.71
clicks_homepage                       3365        10.32
clicks_resource                       3365        10.32
clicks_page                           3365        10.32
clicks_questionnaire                  3365        10.32
clicks_quiz                           3365        10.32
clicks_repeatactivity                 3365        10.32
clicks_early                          3365        10.32
clicks_sharedsubpage                  336

In [40]:
# Fill missing categorical values with "Unknown" for feature columns only
# (excluding key columns and labels)
print("\nFilling missing categorical values with 'Unknown'...")

# Identify categorical columns (exclude numeric and key columns)
key_cols = ['id_student', 'code_module', 'code_presentation']
label_cols = ['final_result', 'at_risk']

# For supervised features
cat_cols_sup = features_supervised.select_dtypes(include=['object']).columns.tolist()
cat_cols_sup = [c for c in cat_cols_sup if c not in key_cols + label_cols]
for col in cat_cols_sup:
    features_supervised[col] = features_supervised[col].fillna('Unknown')

# For unsupervised features
cat_cols_unsup = features_unsupervised.select_dtypes(include=['object']).columns.tolist()
cat_cols_unsup = [c for c in cat_cols_unsup if c not in key_cols]
for col in cat_cols_unsup:
    features_unsupervised[col] = features_unsupervised[col].fillna('Unknown')

print("Categorical missing values filled.")



Filling missing categorical values with 'Unknown'...
Categorical missing values filled.


## 7. Save Feature Files


In [44]:
# Save supervised features
print("Saving supervised features...")

# Try to use parquet format, fallback to CSV if there are any issues
try:
    features_supervised.to_parquet('./data/features_supervised.parquet', index=False, engine='pyarrow')
    print(f"Saved: ./data/features_supervised.parquet ({features_supervised.shape})")
except (ImportError, ModuleNotFoundError) as e:
    print("pyarrow not available. Attempting to install...")
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "--quiet"])
        features_supervised.to_parquet('./data/features_supervised.parquet', index=False, engine='pyarrow')
        print(f"Saved: ./data/features_supervised.parquet ({features_supervised.shape})")
    except Exception as install_error:
        print(f"Could not install/use pyarrow: {install_error}")
        print("Falling back to CSV format...")
        features_supervised.to_csv('./data/features_supervised.csv', index=False)
        print(f"Saved: ./data/features_supervised.csv ({features_supervised.shape})")
except Exception as e:
    # Catch any other errors (like ArrowKeyError, compatibility issues, etc.)
    print(f"Error saving as parquet: {type(e).__name__}: {e}")
    print("Falling back to CSV format...")
    features_supervised.to_csv('./data/features_supervised.csv', index=False)
    print(f"Saved: ./data/features_supervised.csv ({features_supervised.shape})")

# Save unsupervised features
print("\nSaving unsupervised features...")

try:
    features_unsupervised.to_parquet('./data/features_unsupervised.parquet', index=False, engine='pyarrow')
    print(f"Saved: ./data/features_unsupervised.parquet ({features_unsupervised.shape})")
except (ImportError, ModuleNotFoundError) as e:
    print("pyarrow not available. Attempting to install...")
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "--quiet"])
        features_unsupervised.to_parquet('./data/features_unsupervised.parquet', index=False, engine='pyarrow')
        print(f"Saved: ./data/features_unsupervised.parquet ({features_unsupervised.shape})")
    except Exception as install_error:
        print(f"Could not install/use pyarrow: {install_error}")
        print("Falling back to CSV format...")
        features_unsupervised.to_csv('./data/features_unsupervised.csv', index=False)
        print(f"Saved: ./data/features_unsupervised.csv ({features_unsupervised.shape})")
except Exception as e:
    # Catch any other errors (like ArrowKeyError, compatibility issues, etc.)
    print(f"Error saving as parquet: {type(e).__name__}: {e}")
    print("Falling back to CSV format...")
    features_unsupervised.to_csv('./data/features_unsupervised.csv', index=False)
    print(f"Saved: ./data/features_unsupervised.csv ({features_unsupervised.shape})")


Saving supervised features...
Error saving as parquet: ArrowKeyError: A type extension with name pandas.period already defined
Falling back to CSV format...
Saved: ./data/features_supervised.csv ((32593, 46))

Saving unsupervised features...
Error saving as parquet: ArrowKeyError: A type extension with name pandas.period already defined
Falling back to CSV format...
Saved: ./data/features_unsupervised.csv ((32593, 44))


In [47]:
# Create and save feature dictionary
print("Creating feature dictionary...")

# Combine columns from both datasets (supervised has more)
all_columns = set(features_supervised.columns.tolist())

feature_dict = []
for col in sorted(all_columns):
    description = ""
    
    # Key columns
    if col in ['id_student', 'code_module', 'code_presentation']:
        description = f"Key identifier: {col}"
    # Labels
    elif col == 'final_result':
        description = "Target variable: Final result (Pass, Distinction, Fail, Withdrawn)"
    elif col == 'at_risk':
        description = "Binary target: 1 if Fail/Withdrawn, 0 if Pass/Distinction"
    # Assessment features
    elif 'score' in col.lower():
        description = f"Assessment score feature: {col}"
    elif 'submission' in col.lower():
        description = f"Submission-related feature: {col}"
    elif 'assessment' in col.lower():
        description = f"Assessment-related feature: {col}"
    # VLE features
    elif 'click' in col.lower():
        description = f"VLE engagement feature (clicks): {col}"
    elif 'active' in col.lower():
        description = f"VLE engagement feature (activity): {col}"
    # Demographics (optional, but included if present)
    elif col in ['gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability']:
        description = f"Demographic feature: {col}"
    # Other
    else:
        description = f"Feature: {col}"
    
    feature_dict.append({
        'column_name': col,
        'description': description,
        'in_supervised': 'Yes' if col in features_supervised.columns else 'No',
        'in_unsupervised': 'Yes' if col in features_unsupervised.columns else 'No'
    })

feature_dict_df = pd.DataFrame(feature_dict)
feature_dict_df.to_csv('./reports/feature_dictionary.csv', index=False)
print(f"Saved: ./reports/feature_dictionary.csv ({len(feature_dict_df)} features)")

print("\n=== Feature Engineering Complete ===")
print(f"Supervised features: {features_supervised.shape}")
print(f"Unsupervised features: {features_unsupervised.shape}")
print(f"Feature dictionary: {len(feature_dict_df)} features")


Creating feature dictionary...
Saved: ./reports/feature_dictionary.csv (46 features)

=== Feature Engineering Complete ===
Supervised features: (32593, 46)
Unsupervised features: (32593, 44)
Feature dictionary: 46 features


## 8. Model Training and Evaluation


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("Model training libraries imported.")


In [ ]:
# Prepare data for modeling
print("Preparing data for modeling...")

# Use supervised features
df = features_supervised.copy()

# Select features (exclude keys and labels)
exclude_cols = ['id_student', 'code_module', 'code_presentation', 'final_result', 'at_risk']
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Separate features and target
X = df[feature_cols].copy()
y = df['at_risk'].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")


In [ ]:
# Encode categorical features
print("Encoding categorical features...")

# Identify categorical and numeric columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

# Create encoded dataframe
X_encoded = X[numeric_cols].copy()

# Label encode categorical variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Fill any remaining NaN values with median for numeric, mode for categorical
for col in numeric_cols:
    if X_encoded[col].isnull().sum() > 0:
        X_encoded[col].fillna(X_encoded[col].median(), inplace=True)

print(f"Encoded features shape: {X_encoded.shape}")
print("Encoding complete.")


In [ ]:
# Train-test split
print("Splitting data into train and test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train target distribution:\n{y_train.value_counts()}")
print(f"Test target distribution:\n{y_test.value_counts()}")


In [ ]:
# Scale features for GaussianNB (it assumes Gaussian distribution)
print("Scaling features for GaussianNB...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")


In [ ]:
# Initialize models
print("Initializing models...")

models = {
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42, n_estimators=100),
    'GaussianNB': GaussianNB(),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42, n_estimators=100)
}

print(f"Models initialized: {list(models.keys())}")


In [ ]:
# Train and evaluate models
print("Training and evaluating models...\n")

results = {}

for model_name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    
    # Use scaled features for GaussianNB, regular features for others
    if model_name == 'GaussianNB':
        X_train_model = X_train_scaled
        X_test_model = X_test_scaled
    else:
        X_train_model = X_train
        X_test_model = X_test
    
    # Train model
    print("Training...")
    model.fit(X_train_model, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_model)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Store results
    results[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'model': model
    }
    
    # Print results
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    
    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not At Risk', 'At Risk']))

print(f"\n{'='*60}")
print("All models trained and evaluated!")
print(f"{'='*60}")


In [ ]:
# Summary comparison of all models
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'Precision': [results[m]['precision'] for m in results.keys()],
    'Recall': [results[m]['recall'] for m in results.keys()],
    'F1 Score': [results[m]['f1_score'] for m in results.keys()]
})

comparison_df = comparison_df.sort_values('F1 Score', ascending=False)
print("\nModels ranked by F1 Score:")
print(comparison_df.to_string(index=False))

# Save results
comparison_df.to_csv('./reports/model_comparison.csv', index=False)
print(f"\nResults saved to: ./reports/model_comparison.csv")
